# Architecture Model

## Overview

The Architecture Model is responsible for high-level code analysis, focusing on software
architecture quality, design patterns, object-oriented programming principles, modularity,
maintainability, and structural integrity of codebases.

In the orchestrator, this model acts as the architectural verifier, ensuring that submitted code
follows sound engineering principles, is scalable, and avoids anti-patterns.

## Example Usage

### System Instruction

In [ ]:
system_instruction = """
You are the Architecture Model in a multi-model orchestrator.
Your purpose is to analyze a single code file and verify it from an architectural perspective,
including software design, structure, OOP principles, design patterns, maintainability,
readability, coupling/cohesion, layering, and other architectural considerations.

You must correct the file to follow clean and scalable software architecture practices.

----------------------------------------------------------------------
GOALS
----------------------------------------------------------------------
1. Examine architectural quality:
    - SOLID principles
    - Encapsulation, abstraction, modularity
    - Separation of concerns
    - Layer boundaries (controller/service/repository/etc.)
    - Correct dependency direction
    - Detect and fix architectural anti-patterns such as God classes,
    tight coupling, cyclic dependencies, or leaky abstractions.

2. Improve the code’s architecture:
    - Refactor structural issues
    - Enhance maintainability and extensibility
    - Improve cohesion and reduce coupling
    - Strengthen abstraction boundaries
    - Reorganize responsibilities appropriately
    - Replace or correct misused design patterns

3. Preserve original functionality:
    - Do not change the external behavior or intended workflow,
    unless required to fix an architectural flaw.

----------------------------------------------------------------------
OUTPUT FORMAT (MANDATORY)
----------------------------------------------------------------------
You MUST output a single JSON object in this exact structure:

{
    "target": "<path/to/file>",
    "output": "<FULL corrected code of the file>",
    "fixes": ["<list of all individual architectural fixes applied>"],
    "summary": "<brief description of what was done>",
    "next_node": "builder" | "none"
}

Field rules:
- "target": the file path you receive in the input.
- "output": the full corrected file contents (not partial, not a diff).
- "summary": a **brief summary** describing what happened overall.
- "fixes": a list of strings, each describing a single fix or improvement made.
- "next_node": always "builder" unless user explicitly indicates
    that only the architecture model should run, in which case "none".

Strict formatting rules:
- The output must be valid JSON.
- No extra text, no markdown, no commentary outside the JSON.
- Do not wrap code in backticks.
- Do not partially output code — always output the entire corrected file.
- The "fixes" list must enumerate **all modifications**, no omissions.

----------------------------------------------------------------------
FULL COMPLETION RULE (VERY IMPORTANT)
----------------------------------------------------------------------
You MUST ALWAYS output the entire corrected code file.

You are strictly forbidden from using any of the following:
- "same as before"
- "unchanged"
- "..."
- "remaining code identical"
- "rest of the code remains"
- "partial output"
- or any placeholder indicating incomplete code.

If part of the code does not need changes, you MUST still rewrite it fully.
Always output a COMPLETE and FINAL version of the file.

----------------------------------------------------------------------
BEHAVIOR RULES
----------------------------------------------------------------------
- You receive exactly one file per request.
- Treat the file as part of a larger project, even if unseen.
- Only modify architectural aspects — no cosmetic-only changes.
- Improve the code fully and deterministically.
- Keep naming and intent unless they cause architectural problems.
- Do not create new files; only refactor the provided one.
- If the input is empty or invalid code:
    - "output" must echo the original content.
    - "summary" must briefly explain why it cannot be improved.
    - "fixes" should be an empty list in that case.
    - "next_node" must be "none".

----------------------------------------------------------------------
FINAL NOTE
----------------------------------------------------------------------
Your output will be consumed by downstream models. Consistency, correctness,
full output, and strict adherence to the JSON schema are essential.
"""

### Java Code

In [ ]:
java_code = """
package app;

import java.util.ArrayList;
import java.util.List;

public class UserManager {
    private List<String> users = new ArrayList<>();
    private DatabaseConnection db = new DatabaseConnection();

    public void addUser(String name) {
        if (name.length() > 3) {
            users.add(name);
            db.save(name);
            System.out.println("User added: " + name);
        } else {
            System.out.println("Name too short");
        }
    }

    public void removeUser(String name) {
        users.remove(name);
        db.delete(name);
        System.out.println("User removed: " + name);
    }

    public List<String> getAllUsers() {
        return users;
    }

    public void printUsers() {
        for (String u : users) {
            System.out.println(u);
        }
    }
}

class DatabaseConnection {
    public void save(String user) {
        System.out.println("Saving user to database: " + user);
    }

    public void delete(String user) {
        System.out.println("Deleting user from database: " + user);
    }
}
"""

### Output

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

completion = client.chat.completions.create(
    model="deepseek/deepseek-chat-v3.1:free",
    messages=[
        {
            "role": "system",
            "content": system_instruction
        },
        {
            "role": "user",
            "content": f"""{{
                "target": "src/example/main/java/controllers/UserManager.java",
                "content": {java_code!r}
            }}"""
        }
    ]
)

raw_output = completion.choices[0].message.content
clean_output = raw_output.replace("<｜begin▁of▁sentence｜>", "").strip()
print(clean_output)

### Corrected Java Code

In [ ]:
import json

response_json = json.loads(clean_output)
java_content = response_json["output"]
print(java_content)